# पाठ 16 - Microsoft Foundry के साथ स्केलेबल एजेंट तैनात करना

इस नोटबुक में आप एक **उत्पादन-तैयार ग्राहक सहायता एजेंट** बनाएंगे काल्पनिक कंपनी **Contoso** के लिए। पिछले पाठों के विपरीत, यहाँ बात एजेंट के तर्क चक्र की नहीं है — बल्कि यह सब कुछ है जो इसे बड़े पैमाने पर चलाने के लिए सुरक्षित बनाता है:

1. **टूल कॉलिंग** — ऑर्डर की जांच और टिकट निर्माण।
2. **RAG** — ज्ञान आधार से नीति सम्बंधित उत्तर।
3. **मेमोरी** — ग्राहकों को वार्तालाप के दौरान याद रखना।
4. **मॉडल रूटिंग** — सरल अनुरोधों को छोटे मॉडल पर भेजना, जटिल अनुरोधों को बड़े मॉडल पर।
5. **प्रतिक्रिया कैशिंग** — पुन: पूछे गए प्रश्नों को बिना मॉडल कॉल के सेवा देना।
6. **मानव अनुमोदन** — एक सीमा से अधिक रिफंड के लिए स्वीकृति की प्रतीक्षा।
7. **मूल्यांकन द्वार** — एक ऑफ़लाइन परीक्षण सेट जो खराब रिलीज़ को रोकता है।
8. **प्रीक्षणीयता** — प्रत्येक अनुरोध के चारों ओर OpenTelemetry ट्रेसिंग।

प्रत्येक अनुभाग स्व-संपूर्ण और चलाने योग्य है। हर पंक्ति पढ़ें — उत्पादन संबंधी मूल बातें जानबूझकर छोटी रखी गई हैं।


## सेटअप

इस नोटबुक को चलाने से पहले, सुनिश्चित करें कि आपके पास:

1. **एक Microsoft Foundry प्रोजेक्ट** जिसमें एक तैनात चैट मॉडल हो (जैसे `gpt-4.1-mini`)।
2. **Azure CLI में लॉग इन किया हुआ हो** — अपने टर्मिनल में `az login` चलाएं।
3. **आवश्यक पर्यावरण चर सेट किए गए हों:**
   - `AZURE_AI_PROJECT_ENDPOINT` — आपका Microsoft Foundry प्रोजेक्ट एंडपॉइंट।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — आपके तैनात मॉडल का नाम।

RAG सेक्शन **Azure AI Search** का उपयोग करता है जब `AZURE_SEARCH_SERVICE_ENDPOINT` और `AZURE_SEARCH_API_KEY` सेट होते हैं, और इसके बिना एक इन-मेमोरी खोज का सहारा लेता है ताकि नोटबुक बिना Search संसाधन के चल सके।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. उपकरण

प्रोडक्शन उपकरण असली सिस्टम के खिलाफ असली काम करते हैं। यहां हम एक आदेश डेटाबेस और एक टिकटिंग सिस्टम को सादे Python फ़ंक्शंस के साथ अनुकरण करते हैं। `@tool` डेकोरेटर उन्हें एजेंट के लिए उपलब्ध कराता है।

ध्यान दें कि `issue_refund` रिफंड के लिए `approval_mode="always_require"` का उपयोग करता है जो एक सीमा से ऊपर है — यह मानव-इन-द-लूप प्रिमिटिव है जिसे हम बाद में लागू करते हैं।


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — पॉलिसी नॉलेज बेस

पॉलिसी सवालों ("आपकी रिटर्न विंडो क्या है?") का जवाब एक प्राधिकृत स्रोत से दिया जाना चाहिए, मॉडल की मेमोरी से नहीं। हम एक छोटे ज्ञान आधार को एक खोज उपकरण के रूप में लपेटते हैं।

उत्पादन में यह **Azure AI Search** है; यहां हम एक इन-मेमोरी कीवर्ड खोज प्रदान करते हैं ताकि नोटबुक कहीं भी चल सके, और जब पर्यावरण चर मौजूद हों तो स्वचालित रूप से Azure AI Search में बदल जाए।


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. मेमोरी

एक सपोर्ट एजेंट जो यह भूल जाता है कि वह किससे बात कर रहा है, वह एक खराब सपोर्ट एजेंट होता है। हम प्रत्येक ग्राहक के लिए एक छोटा प्रोफाइल स्टोर रखते हैं और एजेंट के निर्देशों में एक छोटा सारांश डालते हैं। उत्पादन में यह एक मेमोरी सेवा है (देखें पाठ 13); यहाँ एक dict पैटर्न को दृश्य बनाता है।


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. मॉडल रूटिंग और प्रतिक्रिया कैशिंग

एक ही अनुरोध हैंडलर में जुड़े दो लागत लीवर:

- **रूटिंग**: एक सस्ता पूर्वानुमान वर्गीकर्ता तय करता है कि किसी अनुरोध को छोटे मॉडल की आवश्यकता है या बड़े मॉडल की।
- **कैशिंग**: सामान्यीकृत दोहराए गए प्रश्न सीधे कैश से परोसे जाते हैं बिना मॉडल कॉल के।

यहां वर्गीकर्ता जानबूझकर सरल है। उत्पादन में आप इसे ट्रैफ़िक के खिलाफ मान्य करेंगे और इसे Foundry के मॉडल राउटर से बदल सकते हैं।


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. एजेंट, मानव अनुमोदन, और प्रेक्षणीयता

अब हम ऊपर दिए गए उपकरणों से एजेंट को इकट्ठा करते हैं और प्रत्येक अनुरोध को एक OpenTelemetry स्पैन में लपेटते हैं। `handle_support_request` फ़ंक्शन उत्पादन अनुरोध हैंडलर है: कैश → रूट → ट्रेस → रन → कैश।

मानव अनुमोदन को फ्रेमवर्क द्वारा हैंडल किया जाता है: क्योंकि `issue_refund` का `approval_mode="always_require"` है, इसलिए रन रुक जाता है और एक अनुमोदन अनुरोध प्रस्तुत करता है जिसे हम स्पष्ट रूप से हल करते हैं।


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. मूल्यांकन गेट

यह पाठ से रिलीज गेट है: एक ऑफ़लाइन टेस्ट सेट एजेंट को स्कोर करता है, और परिनियोजन केवल तभी आगे बढ़ता है जब पास दर एक सीमा को पार कर लेती है। यहां स्कोरर एक सरल कीवर्ड-ओवरलैप जांच है ताकि नोटबुक स्व-निहित बना रहे; उत्पादन में आप एक LLM-एज़-जज या एक फ्रेमवर्क मूल्यांकनकर्ता का उपयोग करेंगे (देखें पाठ 10)।


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## इसे एक साथ रखना: एक सिम्युलेटेड रिलीज़

नीचे दिया गया सेल पूरे लूप को दिखाता है जिसे पाठ में वर्णित किया गया है: मूल्यांकन गेट चलाएँ, और केवल तब "डिप्लॉय" करें जब यह पास हो जाए। यह वह पैटर्न है जिसे आप Foundry Agent Service में एजेंट संस्करण को प्रमोट करने से पहले CI में चलाते हैं।


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## सारांश

आपने एक उत्पादन-तैयार ग्राहक सहायता एजेंट तैयार किया है जिसमें हर परिचालन चिंता को जोड़ा गया है:

- **टूल्स, RAG, और मेमोरी** एजेंट को क्षमता और संदर्भ प्रदान करते हैं।
- **मॉडल राउटिंग और कैशिंग** विलंबता और लागत को नियंत्रण में रखते हैं।
- **मानव अनुमोदन** उच्च जोखिम वाली कार्रवाइयों जैसे बड़े रिफंड की रक्षा करता है।
- **मूल्यांकन द्वार** खराब रिलीज़ को जहाज करने से पहले रोकता है।
- **ट्रेसिंग** हर अनुरोध को अवलोकनीय बनाता है।

### चुनौती

इस एजेंट को विस्तारित करें:

1. **मल्टी मॉडल सपोर्ट** — तीसरी "तर्क" परत जोड़ें और बढ़ोतरी/शिकायतें इसे भेजें।
2. **मूल्यांकन द्वार जोड़ें** — `TEST_CASES` को रिफंड-स्वीकृति परिदृश्यों से विस्तार करें और सुनिश्चित करें कि द्वार रिग्रेशन पकड़ता है।
3. **लागत-सचेत राउटिंग जोड़ें** — प्रत्येक अनुरोध की अनुमानित लागत (छोटा बनाम बड़ा बनाम कैश) ट्रैक करें और मिश्रित क्वेरीज़ के बैच के बाद लागत रिपोर्ट प्रिंट करें।

अगले पाठ में आप विपरीत यात्रा करेंगे और Microsoft Foundry Local और Qwen के साथ पूरी तरह से अपनी मशीन पर एक एजेंट चलाएंगे।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
